# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [22]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [23]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [21]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Artificial Intelligence is transforming modern cricket. "
        "AI analyzes player performance and match statistics. "
        "It helps coaches develop better strategies. "
        "AI can also predict injuries and improve player fitness."
    ),

    "document_1": (
        "AI is used in cricket for batting analysis. "
        "It studies a batsman's shots and timing. "
        "It identifies strengths and weaknesses. "
        "Coaches use this information to improve batting performance."
    ),

    "document_2": (
        "AI helps cricket teams analyze bowling performance. "
        "It studies bowling speed, accuracy and variations. "
        "It helps bowlers improve their techniques. "
        "Teams use AI to plan better bowling strategies."
    ),

    "document_3": (
        "AI is useful for cricket match predictions. "
        "It analyzes previous match results and player statistics. "
        "It estimates team performance and winning chances. "
        "This helps teams make informed decisions."
    ),

    "document_4": (
        "AI helps prevent injuries in cricket players. "
        "It monitors player workload and physical performance. "
        "It can identify possible injury risks. "
        "This helps players maintain their fitness."
    )
}

In [24]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


No relevant documents found for that query.


In [19]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time
import os

from google.colab import userdata
genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        '''
       You are an expert AI cricket research assistant.
When you need specific information about batting,
bowling, match predictions, or player injuries,
use the search_documents tool.
Only answer based on information found in the documents.
If the documents do not contain the answer,
clearly say so.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [10]:
import time
from google import genai
from google.genai import types

from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GOOGLE_API_KEY"),
    http_options=types.HttpOptions(timeout=30000),  # 30 seconds
) # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

def search_documents(query: str) -> str:
    docs = [
        "AI analyzes batting data like strike rate, shot placement and shot type.",
        "Computer vision tracks ball and bat movement to study technique.",
    ]
    return docs[0] + " " + docs[1]

MODEL = "gemini-3.5-flash-lite"  # use the ID from AI Studio

def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL, contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

def run_agent(question: str, max_steps: int = 3) -> str:
    print("Question:", question)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        parts = response.candidates[0].content.parts
        calls = [p.function_call for p in parts if p.function_call]

        if not calls:
            print(response.text)
            return response.text

        contents.append(response.candidates[0].content)
        for call in calls:
            print(f"Step {step}: {call.name}({dict(call.args)})")
            result = search_documents(**call.args)  # your own function
            contents.append(types.Content(role="user", parts=[
                types.Part.from_function_response(
                    name=call.name, response={"result": result})]))

    return "Completed"


_ = run_agent("How does AI help analyze batting performance in cricket?")

Question: How does AI help analyze batting performance in cricket?
Step 1: search_documents({'query': 'AI batting performance cricket'})
Artificial Intelligence (AI) helps analyze batting performance in cricket through several advanced methods, primarily using data analytics and computer vision:

* **Analyzing Performance Data:** AI processes traditional and advanced metrics such as strike rate, shot placement, and shot type to evaluate a batter’s overall effectiveness and consistency.
* **Computer Vision and Technique Study:** AI utilizes computer vision to track both ball and bat movement, allowing coaches and analysts to deeply study a batter's technique, swing mechanics, and body positioning during a shot.


In [12]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
   "How does AI help analyze batting performance in cricket?",
"How can AI predict the performance of a bowler?",
"How does AI help coaches improve team strategy?",
"How is AI used in injury prediction in cricket?",
"How does AI help in decision-making during a cricket match?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: How does AI help analyze batting performance in cricket?
Step 1: search_documents({'query': 'cricket batting performance AI analysis'})
Artificial Intelligence (AI) helps analyze batting performance in cricket through several advanced techniques, primarily using computer vision and data analytics:

* **Technique and Biomechanical Analysis:** Computer vision models track the movement of both the bat and the ball in real-time. This allows coaches and analysts to break down a batsman's stance, backlift, swing path, and head position to identify flaws or inconsistencies in their technique.
* **Shot Classification and Placement:** AI algorithms automatically categorize the types of shots a batter plays (e.g., drive, pull, sweep) and map their shot placement. This helps teams understand a batter's scoring zones and preferred areas of the field.
* **Performance Metrics Tracking:** AI processes large volumes of match data—such as strike rates, scoring frequency against different bow

In [13]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)